<a href="https://colab.research.google.com/github/iannickgagnon/notebooks/blob/main/design_pattern_decorator_strategy_visitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

from random import randint, choice

from dataclasses import dataclass

from typing import Protocol, runtime_checkable

# Generic Knapsack object class

In [2]:
@dataclass
class KnapsackObject:
  """
  Generic Knapsack object class.

  Attributes:
    weight (int): Weight of the object.
    value (int): Value of the object.
  """
  weight: int
  value: int

  def __post__init__(self):
    """
    Validate attributes after initialization.
    """
    if self.weight <= 0:
      raise ValueError("Weight must be > 0")

    if self.value < 0:
      raise ValueError("Value cannot be negative")

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Visitor pattern method for double dispatching.
    """
    raise NotImplementedError("The accept method is only implemented in the concrete subclasses.")

# Concrete KnapsckObject classes

In [3]:
class Television(KnapsackObject):
  """
  Concrete KanpsackObject class that corresponds to a television.

  Attributes:
    name (str): Name of the object.
  """
  name:str = "Television"

  def __init__(self, weight: int, value: int):
    """"
    Calls the parent class constructor.
    """
    super().__init__(weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_television(self)


class Vase(KnapsackObject):
  """
  Concrete KnapsackObject class that corresponds to a vase.
  """
  name:str  = "Vase"

  def __init__(self, weight: int, value: int):
    """
    Calls the parent class constructor.
    """
    super().__init__(weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_vase(self)

# Concrete visitors

In [4]:
@runtime_checkable
class KnapsackObjectVisitor(Protocol):
  """
  Visitor interface for KnapsackObject elements.

  Protocol defines a structural interface, which amounts to duck typing
  with type checking.

  That means:
    - A class does NOT need to inherit from `KnapsackObjectVisitor` to be
      considered a valid visitor.
    - It only needs to provide the same methods with compatible signatures.

  Example:

    class MyVisitor:
      def visit_television(self, obj: Television) -> float: ...
      def visit_vase(self, obj: Vase) -> float: ...

    `MyVisitor` is accepted anywhere a `KnapsackObjectVisitor` is expected,
    because it has the right method names/signatures.

  This is useful for the Visitor pattern because you can add new visitors
  (new operations) without modifying existing object classes, and without
  forcing visitors to inherit from a common base class.

  Normally, `Protocol` is meant for static type checkers (mypy/pyright).

  At runtime, Python does not enforce protocol conformance automatically.

  Decorating the Protocol with @runtime_checkable enables runtime checks like:

      isinstance(visitor, KnapsackObjectVisitor)

  With the decorator:
    - `isinstance` will return True if the object has the required attributes
      (here: `visit_television` and `visit_vase`) with a compatible shape.

    - It’s still not perfect "full type checking" at runtime; it’s mainly an
      structural attribute-presence check, not a guarantee about argument
      and return types.

  Without the decorator:
    - isinstance(visitor, KnapsackObjectVisitor) raises a TypeError.

  The `...` is the literal Python `Ellipsis` object.

  In a Protocol, these method bodies are stubs:
    - They say “this method must exist” and define its signature.
    - They do not provide an implementation.

  The type checker reads these stubs as abstract requirements.

  With this Protocol, the `accept()` methods can be typed as:
      def accept(self, visitor: KnapsackObjectVisitor) -> float:
          ...

  Then any new visitor class (InverseWeightVisitor, ValueDensityVisitor, etc.)
  will type-check as long as it defines the required methods with no inheritance
  required.
  """
  def visit_television(self, obj: Television) -> float: ...
  def visit_vase(self, obj: Vase) -> float: ...

class InverseWeightVisitor:
  """
  Returns the inverse of the object's weight.
  """
  def visit_television(self, obj: Television) -> float:
    return 1.0 / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return 1.0 / obj.weight

class ValueDensityVisitor:
  """
  Returns the value density (value/weight) of the object.
  """
  def visit_television(self, obj: Television) -> float:
    return obj.value / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return obj.value / obj.weight

# Data generator

In [5]:
NB_OBJECTS: int = 10

# --- Television

WEIGHT_MIN_TELEVISION_LB: int = 50
WEIGHT_MAX_TELEVISION_LB: int = 300

VALUE_MIN_TELEVISION: int = 50
VALUE_MAX_TELEVISION: int = 300

# --- Vase

WEIGHT_MIN_VASE_LB: int = 10
WEIGHT_MAX_VASE_LB: int = 1000

VALUE_MIN_VASE: int = 10
VALUE_MAX_VASE: int = 1000

# --- List of available objects

objects: list[KnapsackObject] = []
for i in range(NB_OBJECTS):

  obj_factory = choice([(Television,
                         WEIGHT_MIN_TELEVISION_LB,
                         WEIGHT_MAX_TELEVISION_LB,
                         VALUE_MIN_TELEVISION,
                         VALUE_MAX_TELEVISION),
                        (Vase,
                         WEIGHT_MIN_VASE_LB,
                         WEIGHT_MAX_VASE_LB,
                         VALUE_MIN_VASE,
                         VALUE_MAX_VASE)])

  weight_min: int = obj_factory[1]
  weight_max: int = obj_factory[2]
  value_min: int = obj_factory[3]
  value_max: int = obj_factory[4]

  random_weight: int = randint(weight_min, weight_max)
  random_value: int = randint(value_min, value_max)

  random_object: KnapsackObject = obj_factory[0](weight=random_weight,
                                                 value=random_value)

  objects.append(random_object)

  print(f"Added object no.{i + 1}: {random_object}")

Added object no.1: Television(weight=193, value=196)
Added object no.2: Vase(weight=289, value=789)
Added object no.3: Vase(weight=751, value=490)
Added object no.4: Vase(weight=670, value=218)
Added object no.5: Vase(weight=318, value=337)
Added object no.6: Vase(weight=276, value=343)
Added object no.7: Vase(weight=923, value=486)
Added object no.8: Television(weight=286, value=155)
Added object no.9: Vase(weight=787, value=777)
Added object no.10: Vase(weight=458, value=66)


# Do something with the visitors
Calculate the inverse weights of the objects in the list.

In [6]:
inverse_weight_visitor = InverseWeightVisitor()
value_density_visitor = ValueDensityVisitor()

inverse_weights: list[float] = []
value_densities: list[float] = []
for obj in objects:
  inverse_weights.append(obj.accept(inverse_weight_visitor))
  value_densities.append(obj.accept(value_density_visitor))

for value in inverse_weights:
  print(value)

0.0051813471502590676
0.0034602076124567475
0.0013315579227696406
0.0014925373134328358
0.0031446540880503146
0.0036231884057971015
0.0010834236186348862
0.0034965034965034965
0.0012706480304955528
0.002183406113537118
